# 07/05 Tiny Trace-Specialized LSTM — 605.mcf_s-994B

Train/export a replay-ready **742-trainable-parameter** causal LSTM. This is a pre-run experimental design, not a claim that it already beats AMPM. Only keyed ChampSim replay decides that.

**Policy:** dependency ×2 + PC-delta ×1 + global ×1; at most four candidate deltas/event; label lead 4–128 events; LRU dedup 256; held-out policy floor precision ≥0.25. AMPM is the weak normal reference, so the target is to exceed it while making the dependency proposal source explicit.

```mermaid
flowchart LR
 A[Current producer PC,line + past]-->B[static dependency edge vocabulary]
 A-->C[8 causal runtime features]
 B-->D[≤4 bounded candidates]
 C-->E[8-unit LSTM]
 D-->F[5 candidate features]
 E-->G[concat]
 F-->G
 G-->H[8-unit projection]
 H-->I[utility + 5 lead bins]
 I-->J[held-out threshold/top-k + LRU dedup]
 J-->K[rich list + full decision ledger]
```


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO_ROOT = Path('/content/cache_arch')
REPO_URL = os.environ.get('CACHE_ARCH_REPO_URL', 'https://github.com/Angelawoo572/cache_arch.git')
if not REPO_ROOT.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'], check=True)
print(REPO_ROOT)


In [ ]:
TRACE = '605.mcf_s-994B'
RUN_ID = 'tiny_trace_lstm_07_05_seed7'
SEED, MAX_ROWS, EPOCHS, CHUNK_LEN, LEDGER_SCOPE = 7, 0, 8, 1024, 'full'
ARTIFACT_DIR = REPO_ROOT / 'formal_NN_training/artifacts/tiny_trace_lstm_07_05' / TRACE / RUN_ID
oracle = REPO_ROOT / 'formal_NN_training/results/standalone_nn_data/oracle' / (TRACE + '.oracle.csv.gz')
sidecar_root = REPO_ROOT / 'formal_NN_training/data/upload/v3_9_dependency_profiles'
required = [oracle, Path(str(oracle)+'.meta.json'), sidecar_root/'605.mcf_s-994B.v3_9_dependency_profile.csv.gz', sidecar_root/'605.mcf_s-994B.v3_9_dependency_edge_vocab.csv.gz']
missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
if missing: raise FileNotFoundError('Missing required input(s):\n'+'\n'.join(missing))
print('dependency sidecars are present')


In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'formal_NN_training/LSTM'))
from tiny_trace_prefetcher import run_trace
metadata = run_trace(repo_root=REPO_ROOT, trace=TRACE, run_id=RUN_ID, artifact_root=ARTIFACT_DIR, seed=SEED, max_rows=MAX_ROWS, epochs=EPOCHS, chunk_len=CHUNK_LEN, ledger_scope=LEDGER_SCOPE)
print(json.dumps(metadata, indent=2, sort_keys=True))
assert metadata['parameter_count'] == 742 and metadata['parameter_count'] < 1000
assert metadata['ledger_scope'] == 'full'


In [ ]:
import shutil
from google.colab import files
relative_artifact = ARTIFACT_DIR.relative_to(REPO_ROOT)
archive = shutil.make_archive('/content/' + TRACE + '_' + RUN_ID, 'zip', root_dir=str(REPO_ROOT), base_dir=str(relative_artifact))
print('rich list:', metadata['rich_list'])
print('full ledger:', metadata['decision_ledger'])
print('plan:', metadata['replay_plan'])
files.download(archive)


## After download

Unzip the archive from the repository root on Sacramento. Keep the dependency-sidecar provenance and full ledger. Run the combined replay plan and evidence commands in `formal_NN_training/reports/07_05_tiny_trace_lstm_readme.md`.
